In [ ]:
import customtkinter as ctk
from tkinter import ttk, messagebox
import sqlite3
from datetime import datetime
import matplotlib.pyplot as plt
from reportlab.platypus import SimpleDocTemplate, Paragraph
from reportlab.lib.styles import getSampleStyleSheet

ctk.set_appearance_mode("dark")

# ---------------- DATABASE ----------------
conn = sqlite3.connect("coaching.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS students (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT,
    gender TEXT,
    mobile TEXT UNIQUE,
    course TEXT,
    payment INTEGER,
    location TEXT
)
""")

conn.commit()

cursor.execute("""
CREATE TABLE IF NOT EXISTS attendance (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    student_id INTEGER,
    name TEXT,
    status TEXT,
    date TEXT,
    time TEXT,
    status TEXT
)
""")
conn.commit()

# ---------------- APP ----------------
app = ctk.CTk()
app.geometry("1000x600")
app.title("Coaching ERP Premium")

# ---------------- SIDEBAR ----------------
sidebar = ctk.CTkFrame(app, width=200)
sidebar.pack(side="left", fill="y")

main = ctk.CTkFrame(app)
main.pack(side="right", expand=True, fill="both")

def clear():
    for w in main.winfo_children():
        w.destroy()

# ---------------- HOME ----------------
def home():
    clear()
    ctk.CTkLabel(main, text="Welcome to Coaching ERP", font=("Arial", 24)).pack(pady=50)

# ---------------- ADD STUDENT ----------------
def add_student():
    clear()

    name = ctk.CTkEntry(main, placeholder_text="Name")
    name.pack(pady=5)

    gender = ctk.CTkComboBox(main, values=["Male","Female"])
    gender.pack(pady=5)

    mobile = ctk.CTkEntry(main, placeholder_text="Mobile")
    mobile.pack(pady=5)

    course = ctk.CTkComboBox(main, values=["Data Analyst","Data Science","ML","AI"])
    course.pack(pady=5)

    payment = ctk.CTkEntry(main, placeholder_text="Payment")
    payment.pack(pady=5)

    location = ctk.CTkEntry(main, placeholder_text="Location")
    location.pack(pady=5)

    def save():
        try:
            cursor.execute("""
            INSERT INTO students (name, gender, mobile, course, payment, location)
            VALUES (?, ?, ?, ?, ?, ?)
            """, (
                name.get(), gender.get(), mobile.get(),
                course.get(), int(payment.get()), location.get()
            ))
            conn.commit()

            generate_invoice(name.get(), course.get(), payment.get())

            messagebox.showinfo("Success", "Student Added + Invoice Created")

        except Exception as e:
            messagebox.showerror("Error", f"Error: {e}")

    ctk.CTkButton(main, text="Save + Invoice", command=save).pack(pady=10)

# ---------------- VIEW STUDENTS ----------------
def view_students():
    clear()

    tree = ttk.Treeview(main, columns=("ID","Name","Mobile","Course","Payment"), show="headings")
    for col in tree["columns"]:
        tree.heading(col, text=col)
        tree.column(col, anchor="center")

    tree.pack(fill="both", expand=True, pady=10)

    def load():
        tree.delete(*tree.get_children())
        for row in cursor.execute("SELECT id,name,mobile,course,payment FROM students"):
            tree.insert("", "end", values=row)

    load()

    # -------- DELETE --------
    def delete_student():
        selected = tree.selection()
        if not selected:
            messagebox.showerror("Error", "Select a student")
            return

        values = tree.item(selected[0])["values"]
        student_id = values[0]

        confirm = messagebox.askyesno("Confirm", "Delete this student?")
        if confirm:
            cursor.execute("DELETE FROM students WHERE id=?", (student_id,))
            conn.commit()
            load()
            messagebox.showinfo("Deleted", "Student removed")

    ctk.CTkButton(main, text="Delete Selected", command=delete_student).pack(pady=5)

# ---------------- INVOICE ----------------
def generate_invoice(name, course, payment):
    doc = SimpleDocTemplate(f"{name}_invoice.pdf")
    styles = getSampleStyleSheet()

    content = [
        Paragraph(f"Student Name: {name}", styles["Normal"]),
        Paragraph(f"Course: {course}", styles["Normal"]),
        Paragraph(f"Payment: ₹{payment}", styles["Normal"]),
        Paragraph(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", styles["Normal"])
    ]

    doc.build(content)

# ---------------- CHART ----------------
def show_chart():
    clear()

    cursor.execute("SELECT course, COUNT(*) FROM students GROUP BY course")
    data = cursor.fetchall()

    if not data:
        messagebox.showinfo("Info", "No data available")
        return

    courses = [x[0] for x in data]
    counts = [x[1] for x in data]

    plt.figure()
    plt.bar(courses, counts)
    plt.title("Students per Course")
    plt.show()


# ---------------- ATTENDANCE ----------------
def attendance():
    clear()

    ctk.CTkLabel(main, text="Attendance System", font=("Arial", 20)).pack(pady=10)

    frame = ctk.CTkScrollableFrame(main)
    frame.pack(fill="both", expand=True)

    attendance_vars = {}

    # Fetch students from DB
    cursor.execute("SELECT id, name FROM students")
    students = cursor.fetchall()

    if not students:
        messagebox.showinfo("Info", "No students available")
        return

    for student in students:
        sid, name = student

        row = ctk.CTkFrame(frame)
        row.pack(fill="x", pady=5, padx=10)

        ctk.CTkLabel(row, text=name, width=200).pack(side="left", padx=10)

        var = ctk.StringVar(value="Present")
        attendance_vars[sid] = (name, var)

        dropdown = ctk.CTkComboBox(row, values=["Present", "Absent"], variable=var, width=120)
        dropdown.pack(side="right", padx=10)

    # -------- SAVE ATTENDANCE --------
    def save_attendance():
        now = datetime.now()
        date = now.strftime("%Y-%m-%d")
        time = now.strftime("%H:%M:%S")

        for sid, (name, var) in attendance_vars.items():
            cursor.execute("""
                INSERT INTO attendance (student_id, name, status, date, time)
                VALUES (?, ?, ?, ?, ?)
            """, (sid, name, var.get(), date, time))

        conn.commit()
        messagebox.showinfo("Success", "Attendance Saved")

    ctk.CTkButton(main, text="Save Attendance", command=save_attendance).pack(pady=10)

    # -------- VIEW ATTENDANCE --------
    def view_attendance():
        clear()

        ctk.CTkLabel(main, text="Attendance Records", font=("Arial", 20)).pack(pady=10)

        tree = ttk.Treeview(main, columns=("Name","Status","Date","Time"), show="headings")

        for col in ("Name","Status","Date","Time"):
            tree.heading(col, text=col)
            tree.column(col, anchor="center")

        tree.pack(fill="both", expand=True)

        cursor.execute("SELECT name, status, date, time FROM attendance")
        for row in cursor.fetchall():
            tree.insert("", "end", values=row)

    ctk.CTkButton(main, text="View Attendance", command=view_attendance).pack(pady=5)

# ---------------- SIDEBAR BUTTONS ----------------
ctk.CTkButton(sidebar, text="Home", command=home).pack(pady=10)
ctk.CTkButton(sidebar, text="Add Student", command=add_student).pack(pady=10)
ctk.CTkButton(sidebar, text="View Students", command=view_students).pack(pady=10)
ctk.CTkButton(sidebar, text="Analytics Chart", command=show_chart).pack(pady=10)
ctk.CTkButton(sidebar, text="Attendance", command=attendance).pack(pady=10)


# ---------------- LOAD DEFAULT SCREEN ----------------
home()

# ---------------- RUN ----------------
app.mainloop()